In [1]:
# 01_probability_error.ipynb
# Level 1 (Theorem 1): score and probability perturbation under lattice
# rounding. Verifies the worst-case bound P*Delta/2, the typical scale
# Delta*sqrt(P/12), and the p(1-p) attenuation of the probability error.

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "axes.grid": True,
                     "grid.color": "0.9", "font.size": 9, "axes.edgecolor": "0.3"})
GREYS = ["0.0", "0.3", "0.5", "0.65"]; LS = ["-", "--", "-.", ":"]; MK = ["o", "s", "^", "D"]
Path("../results/figures").mkdir(parents=True, exist_ok=True)
Path("../results/tables").mkdir(parents=True, exist_ok=True)
def _save(fig, name):
    fig.savefig(f"../results/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"../results/figures/{name}.pdf", bbox_inches="tight")

df = pd.read_parquet("../data/synthetic_population.parquet")
ccols = [c for c in df.columns if c.startswith("c")]
C = df[ccols].to_numpy(); f = df["f"].to_numpy(); p = df["p"].to_numpy(); y = df["y"].to_numpy()
P = len(ccols); b0 = float((f - C.sum(1)).mean())
def quant(A, D): return D * np.round(A / D)
def fq(D): return b0 + quant(C, D).sum(1)

Deltas = np.array([0.01, 0.02, 0.05, 0.1, 0.2, 0.4])
rows = []
for D in Deltas:
    e = fq(D) - f
    pq = 1.0 / (1.0 + np.exp(-fq(D)))
    rows.append(dict(Delta=D, max_abs_e=np.abs(e).max(), worst=P * D / 2,
                     std_e=e.std(), typical=D * np.sqrt(P / 12),
                     prob_mad=np.abs(pq - p).mean()))
tab = pd.DataFrame(rows)
tab.to_csv("../results/tables/01_probability_error.csv", index=False)
print(tab.round(4).to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
ax[0].plot(tab.Delta, tab.worst, ls="--", color="0.0", marker="s", label=r"worst case $P\Delta/2$")
ax[0].plot(tab.Delta, tab.max_abs_e, ls="-", color="0.3", marker="o", label=r"observed $\max|e|$")
ax[0].plot(tab.Delta, tab.typical, ls="-.", color="0.5", marker="^", label=r"typical $\Delta\sqrt{P/12}$")
ax[0].plot(tab.Delta, tab.std_e, ls=":", color="0.6", marker="D", label=r"observed $\mathrm{std}(e)$")
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel(r"lattice spacing $\Delta$"); ax[0].set_ylabel("log-odds perturbation")
ax[0].legend(fontsize=7)

# cleaner right panel: light scatter plus a single smooth theoretical envelope
D = 0.2
pq = 1.0 / (1.0 + np.exp(-fq(D)))
idx = np.random.default_rng(0).choice(len(p), 2000, replace=False)
ax[1].scatter(p[idx], np.abs(pq - p)[idx], s=3, color="0.7", alpha=0.5, label="observed")
pg = np.linspace(float(p.min()), float(p.max()), 200)
ax[1].plot(pg, pg * (1 - pg) * (P * D / 2), color="0.0", lw=1.3,
           label=r"$p(1-p)\,P\Delta/2$ envelope")
ax[1].set_xlabel(r"continuous probability $p$"); ax[1].set_ylabel(r"$|\Delta p|$ at $\Delta=0.2$")
ax[1].legend(fontsize=7)
fig.tight_layout(); _save(fig, "01_probability_error"); plt.close(fig)

 Delta  max_abs_e  worst  std_e  typical  prob_mad
  0.01     0.0508    0.1 0.0143   0.0129    0.0007
  0.02     0.0924    0.2 0.0228   0.0258    0.0010
  0.05     0.2644    0.5 0.0648   0.0645    0.0029
  0.10     0.5078    1.0 0.1214   0.1291    0.0053
  0.20     1.0779    2.0 0.2831   0.2582    0.0141
  0.40     2.3107    4.0 0.7293   0.5164    0.0359
